## Train brain-to-voice decoder for instantaneous voice synthesis

Maitreyee Wairagkar, 2025 

In [ ]:
import os
import random
import yaml

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from lib import dataloader, training_utils, model_utils

In [2]:
#=============================================
# Load configs
#=============================================

config_path = "configs/config.yaml"

with open(config_path) as f:
    cfg = yaml.safe_load(f)

## Load neural data

In [3]:
#=============================================
# Assemble all training and validation trials
#=============================================

training_dat, validation_dat, sessions, session_layers = dataloader.load_all_data(cfg)

Number of sessions: 36
['t15_day025', 't15_day027', 't15_day032', 't15_day034', 't15_day039', 't15_day041', 't15_day046', 't15_day048', 't15_day067', 't15_day069', 't15_day074', 't15_day076', 't15_day081', 't15_day088', 't15_day090', 't15_day095', 't15_day097', 't15_day102', 't15_day104', 't15_day109', 't15_day110', 't15_day123', 't15_day132', 't15_day137', 't15_day139', 't15_day144', 't15_day146', 't15_day153', 't15_day165', 't15_day172', 't15_day174', 't15_day179', 't15_day181', 't15_day186', 't15_day188', 't15_day193']
Loaded 312 trials from session t15_day025
Loaded 406 trials from session t15_day027
Loaded 275 trials from session t15_day032
Loaded 224 trials from session t15_day034
Loaded 131 trials from session t15_day039
Loaded 194 trials from session t15_day041
Loaded 288 trials from session t15_day046
Loaded 293 trials from session t15_day048
Loaded 10 trials from session t15_day067
Loaded 216 trials from session t15_day069
Loaded 210 trials from session t15_day074
Loaded 248 

In [4]:
#=============================================
# Get batch trial splits
#=============================================

batch_size = cfg["training"]["batch_size"]

train_batches_trials = training_utils.make_batches_sessionwise(training_dat, batch_size, verbose=False)
val_batches_trials = training_utils.make_batches_sessionwise(validation_dat, batch_size, verbose=False)
print(f"Train batches: {len(train_batches_trials)}  |  Val batches: {len(val_batches_trials)}")

# Shuffle batches
random.shuffle(train_batches_trials)
random.shuffle(val_batches_trials)

# Make a token batch to set the model parameters/shapes
token_batch_X, token_batch_Y, token_sess_num = training_utils.make_single_batch(
    training_dat, train_batches_trials[0], batch_size
)
print(f"Token batch X:{token_batch_X.shape}  Y:{token_batch_Y.shape}  sess:{token_sess_num.shape}")

Train batches: 32480  |  Val batches: 42
Token batch X:(1024, 60, 512)  Y:(1024, 20)  sess:(1024, 1)


## Train the brain-to-voice model

In [ ]:
#=============================================
# Build brain-to-voice model 
#=============================================

n_sessions = session_layers[-1] + 1
b2voice_model = model_utils.build_model(cfg, token_batch_X, token_batch_Y, token_sess_num, n_sessions)

# Optionally load previous weights for initialization 
model_utils.load_model_weights(b2voice_model, cfg["paths"].get("load_model_path"))

# Note: the model was trained iteratively for each new session, starting from the previous session's weights.

In [ ]:
#=============================================
# Train brain-to-voice model
#=============================================

os.makedirs(cfg["paths"]["model_path"], exist_ok=True)
save_model_path = os.path.join(cfg["paths"]["model_path"], cfg["paths"]["save_model_name"])
print(f"Trained model output directory: {cfg['paths']['model_path']}, model name: {cfg['paths']['save_model_name']}")

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=cfg["training"]["early_stopping_patience"],
    verbose=0,
    mode="min",
)
checkpoint = ModelCheckpoint(
    save_model_path,
    save_best_only=True,
    monitor="val_loss",
    mode="min",
)

history = b2voice_model.fit(
    training_utils.batch_generator_on_the_fly(train_batches_trials, training_dat, batch_size),
    steps_per_epoch=len(train_batches_trials),
    epochs=cfg["training"]["epochs"],
    verbose=1,
    validation_data=training_utils.batch_generator_on_the_fly(
        val_batches_trials, validation_dat, batch_size
    ),
    validation_steps=len(val_batches_trials),
    callbacks=[early_stopping, checkpoint],
)

In [ ]:
# Optionally save final weights (final weights often lead to better performance)

b2voice_model.save(save_model_path, save_format="tf")  
print(f"Final model saved to: {save_model_path}")

In [ ]:
#==================================================================================
# Save the model for real-time inference with the required session embedding layer
#==================================================================================
# preselecting the day embedding layer reduces inference time

session_embed_layer = session_layers[-1]  # Use the last session embedding layer for inference
model_utils.save_inference_model(b2voice_model, token_batch_X, save_model_path, day_embed_layer=session_embed_layer)


In [ ]:
# Test the model using inference.ipynb to synthesize voice from held-out neural data